# EDA Notebook (to be adapted into Python module form)

Original EDA from Jupyter notebook

Module version uses `load_gdp()`, `region_mask_and_owners()`, `cell_index()` and `per_cell_counts()` instead of redoing that work from scratch, and is in `src/inference/sparsity_eda.py`. The raw notebook is not wired into the pipeline and isn't run, but is a great reference tool.

In [ ]:
# Fresh runtime. Run this first, alone.
import numpy as np
from clouddrift import datasets

ds = datasets.gdp1h()
print(ds)  # confirm the actual variable names — should be "lat" and "lon"

In [ ]:
# Cell 2: pull just lat/lon (obs-dim, ~789MB each) — nothing else
lon = ds["lon"].values
lat = ds["lat"].values
print("lon range:", lon.min(), lon.max())  # confirms the convention: -180/180 vs 0/360

In [ ]:
# Why this box defined below: each edge is set to include one real ocean feature,
# with a small buffer so it's not sitting right on the boundary.
# South (42S) = past Brazil-Malvinas Confluence (~38S)
# North (4S)  = past Angola's oil basins, short of the fast equatorial currents
# West (60W)  = past the confluence (~53-55W), clear of the S. American shelf
# East (20E)  = covers Benguela upwelling + Cape region (Agulhas leakage)

# This box is specific to our Benguela region, but the FUNCTION is not —
# it takes lat/lon bounds as input, so swapping to a different region later
# just means changing these 4 numbers, not rewriting any code.

In [ ]:
# Cell 3: the region-agnostic filter function
# Handles both longitude conventions and boxes that cross the dateline,
# so this same function works unchanged for any region you swap in later.
def in_box(lon, lat, lon_min, lon_max, lat_min, lat_max):
    lon_norm = ((lon + 180) % 360) - 180  # force into -180..180 regardless of source convention
    if lon_min <= lon_max:
        lon_ok = (lon_norm >= lon_min) & (lon_norm <= lon_max)
    else:  # box wraps the dateline — not our case here, but keeps the function general
        lon_ok = (lon_norm >= lon_min) | (lon_norm <= lon_max)
    return lon_ok & (lat >= lat_min) & (lat <= lat_max)

# The actual Benguela + downstream box decided earlier in this project
region = {"name": "benguela_south_atlantic", "lat_min": -42, "lat_max": -4, "lon_min": -60, "lon_max": 20}

mask = in_box(lon, lat, region["lon_min"], region["lon_max"], region["lat_min"], region["lat_max"])
n_obs = int(mask.sum())
print(f"observations in box: {n_obs:,}")

In [ ]:
# Cell 4: count distinct drifters without ever building a full per-obs ID array
rowsize = ds["rowsize"].values          # tiny — one integer per drifter (19,396 values)
starts = np.concatenate([[0], np.cumsum(rowsize)[:-1]])  # start index of each drifter's obs block
per_traj_hits = np.add.reduceat(mask, starts)            # obs-in-box count, per drifter
n_drifters = int((per_traj_hits > 0).sum())

print(f"region: {region['name']}")
print(f"observations: {n_obs:,}")
print(f"distinct drifters: {n_drifters:,}")

In [ ]:
# Cell 5: pull out just the in-box readings, and figure out which drifter owns each one
# (without ever building a 197-million-long "owner" list — much lighter than before)

idx_true = np.flatnonzero(mask)          # positions of the ~19.27M in-box readings
lon_box = lon[idx_true]
lat_box = lat[idx_true]

cum_rowsize = np.cumsum(rowsize)          # where each drifter's block of readings ends
owner = np.searchsorted(cum_rowsize, idx_true, side="right")  # which drifter each reading belongs to

print(f"in-box readings: {len(idx_true):,}")
print(f"owners resolved: {len(owner):,}")

In [ ]:
# Cell 6: sort every in-box reading into a 1-degree grid cell
step = 1.0
lat_edges = np.arange(region["lat_min"], region["lat_max"] + step, step)
lon_edges = np.arange(region["lon_min"], region["lon_max"] + step, step)
n_lat_bins = len(lat_edges) - 1
n_lon_bins = len(lon_edges) - 1

lat_idx = np.clip(np.digitize(lat_box, lat_edges) - 1, 0, n_lat_bins - 1)
lon_idx = np.clip(np.digitize(lon_box, lon_edges) - 1, 0, n_lon_bins - 1)

print(f"grid size: {n_lat_bins} x {n_lon_bins} = {n_lat_bins * n_lon_bins:,} cells")

In [ ]:
# Cell 7: count DISTINCT drifters per cell (not just readings)
# Trick: pack (cell, drifter) into one number, dedupe, then count how many
# unique drifters landed in each cell — avoids any Python-level loop over 1,975 drifters.
cell_id = lat_idx.astype(np.int64) * n_lon_bins + lon_idx
n_drifters_total = len(rowsize)
combo_key = cell_id * n_drifters_total + owner.astype(np.int64)

unique_combos = np.unique(combo_key)           # each (cell, drifter) pair kept once
cells_of_unique = unique_combos // n_drifters_total
drifters_per_cell = np.bincount(cells_of_unique, minlength=n_lat_bins * n_lon_bins)
density = drifters_per_cell.reshape(n_lat_bins, n_lon_bins)

print(f"max drifters in a single cell: {density.max()}")
print(f"empty cells: {(density == 0).sum()} out of {density.size}")

max drifters in a single cell: 181
empty cells: 702 out of 3040


In [ ]:
# Cell 8: plot it
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.pcolormesh(lon_edges, lat_edges, density, shading="auto", cmap="viridis")
plt.colorbar(label="distinct drifters per 1° cell")
plt.title("Drifter density — Benguela + South Atlantic downstream box")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()

In [ ]:
# Cell 9: confirm what variables are actually available before pulling them
# (run this first — the names below are my best guess based on typical GDP structure,
# but yours might differ, e.g. "sst" vs "temp")
print(list(ds.data_vars))

In [ ]:
# Cell 10: pull the real values, one variable at a time (proven safe from Cell 2)
# Each is a single ~789MB-2GB array — pull full, then slice to just our region.

time_vals = ds["time"].values[idx_true]
ve_vals   = ds["ve"].values[idx_true]      # eastward velocity
vn_vals   = ds["vn"].values[idx_true]      # northward velocity

print("pulled:", len(time_vals), len(ve_vals), len(vn_vals))

In [ ]:
# Cell 11: assemble the actual data subset — this IS your EDA-ready table
import pandas as pd

df_region = pd.DataFrame({
    "drifter_id": owner,
    "time": time_vals,
    "lat": lat_box,     # already pulled in Cell 5
    "lon": lon_box,
    "ve": ve_vals,
    "vn": vn_vals,
})
df_region["speed"] = np.sqrt(df_region["ve"]**2 + df_region["vn"]**2)

print(df_region.shape)
df_region.head()

In [ ]:
# Cell 12: missing values
print(df_region.isna().sum())
print(f"\ntotal rows: {len(df_region):,}")

In [ ]:
# Cell 13: quick descriptive stats
df_region[["ve", "vn", "speed"]].describe()

In [ ]:
# Cell 13 observations:
#
# - Average eastward/northward velocity (ve, vn) is close to 0.
#   This is expected: water flows both east and west (or north and south) across
#   such a large region, so the directions cancel out when averaged.
#
# - Average speed is 0.257 m/s. This is our best single number for "how fast
#   is the water typically moving" in this region.
#
# - Median speed (0.221 m/s) is lower than average speed (0.257 m/s).
#   This means most readings are on the slower side, but a smaller number of
#   much faster readings are pulling the average up. In other words, speed
#   is NOT evenly spread — it's skewed toward slower values with a "tail" of
#   fast ones.(right skewed)
#
# - Max speed is 3.37 m/s, much higher than typical ocean current speeds
#   we'd expect (lecture examples topped out around 0.7 m/s on average).
#   This needs a closer look: it could be a real strong-current event
#   (e.g. an eddy), OR a data quality issue??? (e.g. a drifter that lost its
#   underwater sea anchor, which is known to cause unreliable readings).

In [ ]:
# Cell 14: distributions of the core variables
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_region["speed"].hist(bins=100, ax=axes[0])
axes[0].set_title("Current speed (m/s)")

df_region["ve"].hist(bins=100, ax=axes[1])
axes[1].set_title("Eastward velocity (ve)")

df_region["vn"].hist(bins=100, ax=axes[2])
axes[2].set_title("Northward velocity (vn)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 observations:
#
# - Speed is skewed: most readings are slow (~0.1-0.2 m/s), with a
#   long tail of rare fast readings.
#
# - Eastward (ve) and northward (vn) velocity are both symmetric
#   around 0 — water moving each direction roughly balances out.
#
# - Speed looks different from ve/vn because speed is always
#   positive (direction doesn't cancel out like it does for ve/vn).

In [ ]:
# Cell 15: observations per year
df_region["year"] = pd.to_datetime(df_region["time"]).dt.year
obs_per_year = df_region.groupby("year").size()

obs_per_year.plot(kind="bar", figsize=(12, 4))
plt.title("Observations per year in the Benguela box")
plt.ylabel("count")
plt.show()

In [ ]:
# Cell 15 observations:
#
# - Almost no data before 2003 — sampling only really starts around then.
#
# - Observations peak around 2007 (~1.48M), then drop and fluctuate
#   between roughly 0.5M-1.3M per year for the rest of the record.
#
# - Sampling is NOT even across the 35-year record. This matters:
#   any average we compute will be mostly based on 2005-2022 data,
#   and comparing early years to later years directly would be
#   misleading (too little early data to trust).

In [ ]:
# Cell 16: time each drifter spends inside the region
track_span = df_region.groupby("drifter_id")["time"].agg(["min", "max", "count"])
track_span["days_in_box"] = (pd.to_datetime(track_span["max"]) - pd.to_datetime(track_span["min"])).dt.days

track_span["days_in_box"].hist(bins=50, figsize=(8, 4))
plt.title("Days spent in box, per drifter")
plt.xlabel("days"); plt.ylabel("number of drifters")
plt.show()

print(f"median days in box: {track_span['days_in_box'].median():.0f}")
print(f"drifters in box < 30 days: {(track_span['days_in_box'] < 30).sum():,} of {len(track_span):,}")

In [ ]:
# Cell 16 observations:
#
# - Median time a drifter spends in the box: 258 days (~8.5 months).
#   So a "typical" drifter does stick around for a reasonably long time.
#
# - Only 279 out of 1,975 drifters (about 14%) leave within 30 days.
#   The other 86% stay for a month or longer.
#
# - Combined with the median, this suggests most of our 1,975 drifters
#   ARE useful for questions needing sustained tracking (e.g. "where
#   does something end up after a year") — it's a minority (~14%) that
#   just clip the edge briefly.

In [ ]:
# Cell 17: 19M points is too many to scatter directly — sample for visualization only
sample = df_region.sample(n=300_000, random_state=42)

plt.figure(figsize=(10, 8))
sc = plt.scatter(sample["lon"], sample["lat"], c=sample["speed"], s=1, cmap="viridis", vmax=sample["speed"].quantile(0.99))
plt.colorbar(sc, label="speed (m/s)")
plt.title("Current speed across the region (sampled)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()

In [ ]:
# Cell 18: raw observation count per cell (not distinct drifters)
obs_per_cell = np.bincount(cell_id, minlength=n_lat_bins * n_lon_bins)
obs_density = obs_per_cell.reshape(n_lat_bins, n_lon_bins)

print(f"max observations in a single cell: {obs_density.max():,}")
print(f"empty cells: {(obs_density == 0).sum()} out of {obs_density.size}")
print(f"median observations per non-empty cell: {int(np.median(obs_density[obs_density > 0])):,}")

max observations in a single cell: 29,359
empty cells: 702 out of 3040
median observations per non-empty cell: 7,133


In [ ]:
# Cell 19: plot both side by side for direct comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

im1 = axes[0].pcolormesh(lon_edges, lat_edges, density, shading="auto", cmap="viridis")
axes[0].set_title("Distinct drifters per cell")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
fig.colorbar(im1, ax=axes[0], label="drifters")

# log scale here because observation counts span a much wider range than drifter counts —
# a handful of long-parked drifters can make one cell's count dwarf the rest
im2 = axes[1].pcolormesh(lon_edges, lat_edges, obs_density, shading="auto", cmap="viridis",
                           norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=obs_density.max()))
axes[1].set_title("Observations per cell (log scale)")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
fig.colorbar(im2, ax=axes[1], label="observations (log scale)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 20: quick check — does the density pattern hold up if restricted to recent years only?
recent_mask = df_region["year"] >= 2010  # or wherever you consider "well-sampled" starting
recent_obs_per_cell = np.bincount(cell_id[recent_mask.values], minlength=n_lat_bins * n_lon_bins)
print(f"observations if we only use 2010+: {recent_mask.sum():,} (vs {len(df_region):,} using all years)")
print(f"empty cells using 2010+ only: {(recent_obs_per_cell.reshape(n_lat_bins, n_lon_bins) == 0).sum()} out of {n_lat_bins * n_lon_bins}")

In [ ]:
# Cell 20 observations:
#
# - Restricting to 2010+ keeps 11,281,893 observations — about 59% of our
#   full 19.27M total. Still a huge number, well above the course benchmark.
#
# - Empty cells go UP slightly: 726 out of 3,040 (vs 702 using all years).
#   So restricting to recent years barely changes coverage — losing 41% of
#   our data only costs us 24 more empty cells.
#
# - This suggests the "extra" pre-2010 data isn't filling in many NEW cells —
#   it's likely adding more readings to cells that were already covered by
#   recent years, rather than covering genuinely different areas.

# Questions for the team:
#
# 1. Do we use ALL 35 years of data, or restrict to a more recent window
#    (e.g. 2010+)? Since empty cells barely change (702 vs 726), coverage
#    isn't the deciding factor — so what is? E.g. do we care more about
#    having MORE data per cell (favors all years) or more CONSISTENT
#    sampling across time (favors recent years only)?
#
# 2. If we use all years, are we okay knowing our transition estimates
#    will lean heavily on 2005-2022 conditions anyway (per Cell 15), even
#    though we're technically using 1994-2022 data?
#
# 3. Does this choice affect the business stream's seasonal forecasting
#    work (Week 6+)? A shorter, more recent window might mean a cleaner
#    "recent seasonal pattern" story vs. a longer but more uneven one.
#
# 4. Should this be a FIXED decision now, or should we test the
#    transition-matrix pipeline both ways later (Week 8 robustness check)
#    and compare results?